# TP4 — Agent Orchestration with LangChain

## Objectives
- Understand LLM-driven agents and their executor loop.
- Use tools: Wikipedia, Tavily, arXiv, Python REPL.
- Create custom tools and compose multi-agent pipelines.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
print('Mistral key:', bool(os.getenv('MISTRAL_API_KEY')))
print('Tavily key: ', bool(os.getenv('TAVILY_API_KEY')))
print('OpenAI key: ', bool(os.getenv('OPENAI_API_KEY')))


## Step 2 — Wikipedia Tool (Ex 2.1, 2.2)


In [ ]:
from langchain.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
print(wikipedia.run('Alan Turing')[:500])

# Ex 2.1: summarise_article function
def summarise_article(topic: str) -> str:
    raw = wikipedia.run(topic)
    return (raw[:500].rstrip() + '...') if len(raw) > 500 else raw

for topic in ['Artificial Intelligence', 'Mistral AI']:
    print(f'--- {topic} ---')
    print(summarise_article(topic))
    print()


## Step 3 — ReAct Agent with Tavily (Ex 3.1, 3.2)


In [ ]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import WikipediaQueryRun
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_mistralai.chat_models import ChatMistralAI

tavily = TavilySearchResults(max_results=2)
wiki   = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
llm    = ChatMistralAI(model='mistral-large-latest', temperature=0)
prompt = hub.pull('amalnuaimi/react-mistral')

# Ex 3.1: two tools
tools = [tavily, wiki]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True,
                         return_intermediate_steps=True)

QUERY = 'Should I bring an umbrella? I travel to Belfort today and tomorrow.'
response = executor.invoke({'input': QUERY, 'chat_history': []})
print('Answer:', response['output'])

# Ex 3.1: inspect intermediate steps
for i, (action, obs) in enumerate(response['intermediate_steps'], 1):
    print(f'Step {i}: tool={action.tool}, input={action.tool_input}')
    print(f'  obs: {str(obs)[:150]}')


## Step 5 — arXiv Scientific Literature Agent (Ex 5.1, 5.2)


In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent, load_tools
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.prompts import PromptTemplate

llm   = ChatMistralAI(model='mistral-large-latest', temperature=0)
tools = load_tools(['arxiv'])

# Ex 5.1: document the tool
print('Tool name       :', tools[0].name)
print('Tool description:', tools[0].description)

# Ex 5.2: structured summary prompt
STRUCTURED_TEMPLATE = """You have access to tools:\n{tools}\n
When summarising a paper use this structure:\n
**Summary**: <2-3 sentence overview>\n
**Key Points**:\n  - <point 1>\n  - <point 2>\n
**Potential Applications**:\n  - <app 1>\n  - <app 2>\n
Format:\nQuestion: {input}\nThought:{agent_scratchpad}
Use tool: [{tool_names}]"""

prompt = PromptTemplate.from_template(STRUCTURED_TEMPLATE)
agent  = create_react_agent(llm, tools, hub.pull('hwchase17/react'))
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
executor.invoke({'input': 'Summarise the paper 1605.08386 in English.'})


## Step 8 — Custom Tools (Ex 8.1, 8.2, 8.3)


In [ ]:
import datetime
from pathlib import Path
from langchain_core.tools import tool, StructuredTool
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

LOG_FILE = Path('operations.log')

def _log(op, detail):
    with open(LOG_FILE, 'a') as f:
        f.write(f'[{datetime.datetime.now().isoformat(timespec="seconds")}] {op}: {detail}\n')

@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiply two integers."""
    r = int(first_int) * int(second_int)  # Ex 8.1: int() conversion
    _log('multiply', f'{first_int}*{second_int}={r}')  # Ex 8.2
    return r

@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    r = int(first_int) + int(second_int)
    _log('add', f'{first_int}+{second_int}={r}')
    return r

@tool
def exponentiate(base: int, exponent: int) -> int:
    """Raise base to exponent."""
    r = int(base) ** int(exponent)
    _log('exponentiate', f'{base}^{exponent}={r}')
    return r

# Ex 8.3: StructuredTool with Pydantic schema
class DivideInput(BaseModel):
    numerator: float = Field(..., description='Number to divide')
    denominator: float = Field(..., description='Divisor (non-zero)')

divide = StructuredTool.from_function(
    func=lambda numerator, denominator: numerator/denominator if denominator != 0 else 'Error: division by zero',
    name='divide', description='Divide two numbers.',
    args_schema=DivideInput,
)

tools = [multiply, add, exponentiate, divide]
llm   = ChatOpenAI(model_name='gpt-4.1', temperature=0)
agent = create_tool_calling_agent(llm, tools, hub.pull('hwchase17/openai-tools-agent'))
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

executor.invoke({'input': 'Raise 3 to the power of 5, multiply by sum of 12 and 3, then square it.'})

print('Log entries:')
print(LOG_FILE.read_text())
